# Application 1 — No-fault divorce reforms and female suicide

Complete local replication notebook for **pydrlpdid 0.7.2**.

Expected Windows directory:

`C:\Users\danie\OneDrive\1 - Pesquisas\0 - DRLPDID\pydrlpdid-0.7.2`

The notebook estimates the eight local specifications reported in the
application over \(h=-10,\ldots,19\), computes state-cluster-robust
inference, and constructs simultaneous Rademacher multiplier bands for
LPDID-RA and DRLPDID-IPT. The optional literature-benchmark block reads
standard errors and confidence intervals directly from `diff_diff`;
it does not reconstruct benchmark inference.

## 0. Configuration

In [ ]:
from pathlib import Path

PROJECT_DIR_WINDOWS = Path(
    r"C:\Users\danie\OneDrive\1 - Pesquisas\0 - DRLPDID\pydrlpdid-0.7.2"
)
DATA_URL = "https://github.com/Daniel-Uhr/data/raw/main/bacon_example.dta"
ALLOW_DATA_DOWNLOAD = True
RUN_EXTERNAL_BENCHMARKS = True
RUN_RICHER_SPECIFICATION = True
CLEAN_OUTPUT_DIRECTORY = True
H_PRE, H_POST = 10, 19
POST_WINDOW = (0, 19)
BASE_PERIOD = -1
WINDOW_TAG = f"h{H_POST}"
BASELINE_COVARIATES = ["asmrh"]
RICHER_COVARIATES_RAW = ["pcinc", "asmrh", "cases"]
RICHER_COVARIATES = [
    f"{column}_z" for column in RICHER_COVARIATES_RAW
]
ALPHA = 0.05
SEED = 123
N_MULTIPLIER = 999

## 1. Environment and package gate

In [ ]:
import hashlib
import importlib.metadata as importlib_metadata
import inspect
import json
import platform
import sys
import warnings
from urllib.request import urlretrieve

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("default")

def locate_root(preferred):
    if (preferred / "src" / "pydrlpdid" / "__init__.py").exists():
        return preferred.resolve()
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (candidate / "src" / "pydrlpdid" / "__init__.py").exists():
            return candidate
    raise FileNotFoundError(
        "Run the notebook from the pydrlpdid-0.7.2 root or update "
        "PROJECT_DIR_WINDOWS."
    )

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

PROJECT_DIR = locate_root(PROJECT_DIR_WINDOWS)
SRC_DIR = PROJECT_DIR / "src"
DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "application_bacon_v072_h19"
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if CLEAN_OUTPUT_DIRECTORY:
    for generated_file in OUTPUT_DIR.iterdir():
        if generated_file.is_file():
            generated_file.unlink()
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import pydrlpdid
from pydrlpdid import DRLPDID, LPDID

if pydrlpdid.__version__ != "0.7.2":
    raise RuntimeError(
        f"Expected pydrlpdid 0.7.2, loaded {pydrlpdid.__version__} "
        f"from {pydrlpdid.__file__}. Restart the kernel."
    )
if PROJECT_DIR not in Path(pydrlpdid.__file__).resolve().parents:
    raise RuntimeError("A different local copy of pydrlpdid was imported.")
PACKAGE_FILE = Path(pydrlpdid.__file__).resolve()
PACKAGE_CORE_FILE = PACKAGE_FILE.parent / "drlpdid.py"
if not PACKAGE_CORE_FILE.exists():
    raise FileNotFoundError(
        f"Could not locate the estimator source at {PACKAGE_CORE_FILE}."
    )

try:
    from diff_diff import CallawaySantAnna, ImputationDiD, SunAbraham, TwoStageDiD
    HAVE_DIFFDIFF = True
    DIFFDIFF_ERROR = None
    try:
        DIFFDIFF_VERSION = importlib_metadata.version("diff-diff")
    except importlib_metadata.PackageNotFoundError:
        DIFFDIFF_VERSION = "unknown"
except Exception as exc:
    HAVE_DIFFDIFF = False
    DIFFDIFF_ERROR = f"{type(exc).__name__}: {exc}"
    DIFFDIFF_VERSION = None

print("Project:", PROJECT_DIR)
print("pydrlpdid:", pydrlpdid.__version__, pydrlpdid.__file__)
print("Estimator source SHA-256:", sha256_file(PACKAGE_CORE_FILE))
print("diff_diff:", DIFFDIFF_VERSION if HAVE_DIFFDIFF else DIFFDIFF_ERROR)
print("Output:", OUTPUT_DIR)

## 2. Data and certified estimation sample

In [ ]:
candidates = [
    DATA_DIR / "bacon_example.csv",
    DATA_DIR / "bacon_example.dta",
    PROJECT_DIR / "bacon_example.csv",
    PROJECT_DIR / "bacon_example.dta",
]
DATA_PATH = next((path for path in candidates if path.exists()), None)
if DATA_PATH is None:
    if not ALLOW_DATA_DOWNLOAD:
        raise FileNotFoundError("Copy bacon_example.dta to the data directory.")
    DATA_PATH = DATA_DIR / "bacon_example.dta"
    urlretrieve(DATA_URL, DATA_PATH)

raw = (
    pd.read_csv(DATA_PATH)
    if DATA_PATH.suffix.lower() == ".csv"
    else pd.read_stata(DATA_PATH, convert_categoricals=False)
)
required = {"stfips", "year", "asmrs", "asmrh", "post", "_nfd"}
missing = sorted(required - set(raw))
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df = raw.copy()
df["year"] = pd.to_numeric(df["year"], errors="raise").astype(int)
df["post"] = pd.to_numeric(df["post"], errors="raise").astype(int)
always_treated = df.groupby("stfips")["post"].transform("min").eq(1)
dropped_always = int(df.loc[always_treated, "stfips"].nunique())
df = df.loc[~always_treated].copy().reset_index(drop=True)
df["g"] = pd.to_numeric(df["_nfd"], errors="coerce").fillna(0).astype(int)
df["id"] = df["stfips"]
df["y"] = pd.to_numeric(df["asmrs"], errors="raise")
df["asmrh"] = pd.to_numeric(df["asmrh"], errors="raise")

missing_richer = sorted(set(RICHER_COVARIATES_RAW) - set(df))
if RUN_RICHER_SPECIFICATION and missing_richer:
    raise ValueError(
        "Missing covariates for the expanded specification: "
        f"{missing_richer}"
    )
scaling_rows = []
for raw_name, model_name in zip(
    RICHER_COVARIATES_RAW, RICHER_COVARIATES
):
    if raw_name not in df:
        continue
    values = pd.to_numeric(df[raw_name], errors="raise")
    if values.isna().any():
        raise ValueError(f"{raw_name} contains missing values.")
    center = float(values.mean())
    scale = float(values.std(ddof=0))
    if not np.isfinite(scale) or scale <= 0:
        raise ValueError(f"{raw_name} has no finite positive scale.")
    df[model_name] = (values - center) / scale
    scaling_rows.append({
        "raw_column": raw_name,
        "model_column": model_name,
        "center": center,
        "scale_ddof0": scale,
    })
covariate_scaling = pd.DataFrame(scaling_rows)
covariate_scaling.to_csv(
    OUTPUT_DIR / "richer_covariate_scaling.csv", index=False
)

expected_post = ((df["g"] > 0) & (df["year"] >= df["g"])).astype(int)
if not df["post"].equals(expected_post):
    raise RuntimeError("post and _nfd do not encode the same absorbing treatment.")

unit_g = df.groupby("id")["g"].first()
audit = {
    "data_path": str(DATA_PATH.resolve()),
    "data_sha256": sha256_file(DATA_PATH),
    "rows": int(len(df)),
    "states": int(df["id"].nunique()),
    "year_min": int(df["year"].min()),
    "year_max": int(df["year"].max()),
    "dropped_initially_treated_states": dropped_always,
    "treated_states": int((unit_g > 0).sum()),
    "never_treated_states": int((unit_g == 0).sum()),
    "cohorts": sorted(int(value) for value in unit_g[unit_g > 0].unique()),
}
(OUTPUT_DIR / "sample_audit.json").write_text(
    json.dumps(audit, indent=2), encoding="utf-8"
)

support_rows = []
for horizon in range(0, max(H_POST, 20) + 1):
    eligible_units = unit_g.loc[
        (unit_g > 0)
        & (unit_g + horizon <= df["year"].max())
        & (unit_g + BASE_PERIOD >= df["year"].min())
    ]
    support_rows.append({
        "horizon": horizon,
        "treated_units": int(len(eligible_units)),
        "treated_cohorts": int(eligible_units.nunique()),
        "cohorts": ",".join(
            map(
                str,
                sorted(int(value) for value in eligible_units.unique()),
            )
        ),
        "included_in_scalar": horizon <= H_POST,
    })
horizon_support = pd.DataFrame(support_rows)
horizon_support.to_csv(
    OUTPUT_DIR / "treated_support_by_horizon.csv", index=False
)

print(json.dumps(audit, indent=2))
display(df[["id", "year", "g", "post", "y", "asmrh"]].head())
display(horizon_support.tail(3))

## 3. Eight local estimators

All specifications use the same requested window and the same
predetermined baseline covariate. The five local ATT estimators use the
common supported stack enforced by `DRLPDID`. LPDID-RW, LPDID-RW+X,
and LPDID-RA are literature benchmarks estimated by `LPDID`. The
formal RA member is reported separately as DRLPDID-RA.

In [ ]:
LOCAL_SPECS = [
    ("LPDID-RW", "lpdid-rw", False),
    ("LPDID-RW+X", "lpdid-rw", True),
    ("LPDID-RA", "lpdid-ra", True),
    ("DRLPDID-RA", "ra", True),
    ("IPW", "ipw", True),
    ("IPT", "ipt", True),
    ("DRLPDID-IPW", "dr-ipw", True),
    ("DRLPDID-IPT", "dr-ipt", True),
]

def fit_local(
    label,
    method,
    use_covariates,
    inference="cluster",
    covariates_override=None,
):
    covariates = (
        list(covariates_override)
        if covariates_override is not None
        else (BASELINE_COVARIATES if use_covariates else None)
    )
    if method == "lpdid-ra":
        return LPDID(
            target_estimand="ra",
            base_period=-1,
            clean_control="not_yet_treated",
            inference=inference,
            n_bootstrap=N_MULTIPLIER,
            bootstrap_weights="rademacher",
            alpha=ALPHA,
            seed=SEED,
            max_pre=H_PRE,
            max_post=H_POST,
            support_policy="supported_subset",
            left_censoring="error",
        ).fit(
            df, outcome="y", unit="id", time="year",
            first_treat="g", covariates=covariates,
        )
    if method == "lpdid-rw":
        if inference != "cluster":
            raise ValueError("LPDID-RW is estimated analytically in this notebook.")
        return LPDID(
            target_estimand="rw",
            base_period=-1,
            clean_control="not_yet_treated",
            inference="cluster",
            alpha=ALPHA,
            max_pre=H_PRE,
            max_post=H_POST,
            support_policy="strict",
            left_censoring="error",
        ).fit(
            df, outcome="y", unit="id", time="year",
            first_treat="g", covariates=covariates,
        )
    return DRLPDID(
        estimation_method=method,
        design="absorbing",
        control_group="not_yet_treated",
        horizons=(-H_PRE, H_POST),
        post_window=POST_WINDOW,
        inference=inference,
        n_bootstrap=N_MULTIPLIER,
        alpha=ALPHA,
        seed=SEED,
    ).fit(
        df, outcome="y", unit="id", time="year",
        first_treat="g", covariates=covariates,
    )

def validate_path(label, result):
    event = result.event_study.sort_values("horizon").reset_index(drop=True)
    expected = set(range(-H_PRE, H_POST + 1))
    observed = set(event["horizon"].astype(int))
    if observed != expected:
        raise RuntimeError(f"{label}: incomplete horizon grid.")
    if event["estimate"].isna().any():
        raise RuntimeError(
            f"{label}: unsupported horizons "
            f"{event.loc[event['estimate'].isna(), 'horizon'].tolist()}."
        )
    base = event.set_index("horizon").loc[-1]
    if not np.allclose(base[["estimate", "se", "ci_lower", "ci_upper"]], 0.0):
        raise RuntimeError(f"{label}: h=-1 is not normalized to zero.")
    return event

def extract_scalar(label, result, event):
    term = (
        f"ATT policy-window [0,{H_POST}]"
        if label not in {"LPDID-RW", "LPDID-RW+X", "LPDID-RA"}
        else "ATT avg"
    )
    selected = result.scalars.loc[result.scalars["term"].eq(term)]
    if selected.empty:
        raise RuntimeError(f"{label}: scalar term {term!r} is absent.")
    row = selected.iloc[0]
    direct = event.loc[event["horizon"].between(0, H_POST), "estimate"].mean()
    if not np.isclose(float(row["estimate"]), float(direct), atol=1e-9):
        raise RuntimeError(f"{label}: scalar and path average differ.")
    return {
        "estimator": label,
        "estimate": float(row["estimate"]),
        "se": float(row["se"]),
        "ci_lower": float(row["ci_lower"]),
        "ci_upper": float(row["ci_upper"]),
        "window": f"h=0,...,{H_POST}",
    }

local_results, local_paths, scalar_rows = {}, {}, []
for label, method, use_covariates in LOCAL_SPECS:
    print("Estimating", label)
    result = fit_local(label, method, use_covariates)
    path = validate_path(label, result)
    local_results[label] = result
    local_paths[label] = path
    scalar_rows.append(extract_scalar(label, result, path))

local_table = pd.DataFrame(scalar_rows)
local_event = pd.concat(
    [path.assign(estimator=label) for label, path in local_paths.items()],
    ignore_index=True,
)
local_table.to_csv(OUTPUT_DIR / "table_local_h19.csv", index=False)
local_table.to_latex(
    OUTPUT_DIR / "table_local_h19.tex",
    index=False,
    float_format=lambda value: f"{value:.3f}",
    caption=(
        "No-fault divorce: local estimators, equal-horizon "
        "average over $h=0,\\ldots,19$"
    ),
    label="tab:bacon-local-h19",
)
local_event.to_csv(OUTPUT_DIR / "local_event_study_h19.csv", index=False)
display(local_table.round(4))

## 4. Formal validation gates

In [ ]:
common_labels = ["DRLPDID-RA", "IPW", "IPT", "DRLPDID-IPW", "DRLPDID-IPT"]
count_columns = [
    "horizon",
    "n_event_rows",
    "n_event_units",
    "n_control_rows",
    "n_control_units",
    "n_reference_dates",
]
reference_counts = local_paths[common_labels[0]][count_columns]
for label in common_labels[1:]:
    pd.testing.assert_frame_equal(
        reference_counts.reset_index(drop=True),
        local_paths[label][count_columns].reset_index(drop=True),
        check_dtype=False,
    )

nested = local_paths["IPT"][["horizon", "estimate"]].merge(
    local_paths["DRLPDID-IPT"][["horizon", "estimate"]],
    on="horizon", suffixes=("_ipt", "_dript"), validate="one_to_one",
)
nested["difference"] = nested["estimate_ipt"] - nested["estimate_dript"]
nested["absolute_difference"] = nested["difference"].abs()
dript_diagnostics = local_results["DRLPDID-IPT"].metadata[
    "nuisance_diagnostics"
]
nested["certified_tolerance"] = nested["horizon"].map(
    lambda h: (
        0.0
        if int(h) == -1
        else dript_diagnostics[int(h)][
            "ipt_dript_identity_tolerance"
        ]
    )
)
nested["retained_or_balance_error"] = nested["horizon"].map(
    lambda h: (
        0.0
        if int(h) == -1
        else dript_diagnostics[int(h)][
            "ipt_nested_or_balance_error"
        ]
    )
)
nested["passes_certified_identity"] = (
    nested["absolute_difference"] <= nested["certified_tolerance"]
)
nested.to_csv(OUTPUT_DIR / "nested_basis_identity_audit.csv", index=False)
maximum_difference = float(
    nested.loc[nested["horizon"].ne(-1), "absolute_difference"].max()
)
if not nested["passes_certified_identity"].all():
    failed = nested.loc[~nested["passes_certified_identity"]]
    raise RuntimeError(
        "IPT–DRLPDID-IPT identity failed at horizons "
        f"{failed['horizon'].astype(int).tolist()}."
    )

rank_rows = []
for horizon in range(-H_PRE, H_POST + 1):
    if horizon == BASE_PERIOD:
        continue
    diagnostic = dript_diagnostics[int(horizon)]
    rank_rows.append({
        "horizon": horizon,
        "ipt_retained_columns": "|".join(
            diagnostic.get("ipt_retained_columns", ())
        ),
        "ipt_dropped_columns": "|".join(
            diagnostic.get("ipt_dropped_columns", ())
        ),
        "or_retained_columns": "|".join(
            diagnostic.get("or_retained_columns", ())
        ),
        "or_dropped_columns": "|".join(
            diagnostic.get("or_dropped_columns", ())
        ),
        "ipt_balance_error": diagnostic.get(
            "ipt_balance_error", np.nan
        ),
        "nested_basis": diagnostic.get(
            "ipt_dript_nested_basis", False
        ),
    })
nuisance_rank_audit = pd.DataFrame(rank_rows)
nuisance_rank_audit.to_csv(
    OUTPUT_DIR / "nuisance_rank_audit.csv", index=False
)

validation = {
    "requested_horizons": [-H_PRE, H_POST],
    "post_window": list(POST_WINDOW),
    "base_period": BASE_PERIOD,
    "formal_estimators_common_supported_stack": True,
    "lpdid_ra_native_dube_sample": True,
    "common_stack_estimators": common_labels,
    "no_propensity_clipping": True,
    "no_fallback_estimator": True,
    "nested_identity_pass": bool(
        nested["passes_certified_identity"].all()
    ),
    "maximum_ipt_dript_absolute_difference": maximum_difference,
    "certified_tolerance_at_maximum": float(
        nested.loc[
            nested["absolute_difference"].idxmax(),
            "certified_tolerance",
        ]
    ),
}
(OUTPUT_DIR / "local_validation.json").write_text(
    json.dumps(validation, indent=2), encoding="utf-8"
)
print("Common-stack gate: passed")
print(
    "IPT–DRLPDID-IPT numerical identity gate: passed; "
    f"maximum difference={maximum_difference:.3e}"
)

## 5. Pointwise intervals and simultaneous bands

In [ ]:
joint_paths = {}
for label, method in [("LPDID-RA", "lpdid-ra"), ("DRLPDID-IPT", "dr-ipt")]:
    result = fit_local(label, method, True, inference="multiplier")
    path = validate_path(label + " multiplier", result)
    original = local_paths[label][["horizon", "estimate", "se", "ci_lower", "ci_upper"]]
    columns = [
        "horizon", "estimate", "sim_ci_lower", "sim_ci_upper"
    ]
    joint = original.merge(
        path[columns].rename(columns={"estimate": "multiplier_estimate"}),
        on="horizon", validate="one_to_one",
    )
    if not np.allclose(joint["estimate"], joint["multiplier_estimate"], atol=1e-10):
        raise RuntimeError(f"{label}: point estimates changed across inference modes.")
    joint.drop(columns="multiplier_estimate", inplace=True)
    joint.loc[joint["horizon"].eq(-1), ["sim_ci_lower", "sim_ci_upper"]] = 0.0
    if joint.loc[joint["horizon"].ne(-1), ["sim_ci_lower", "sim_ci_upper"]].isna().any().any():
        raise RuntimeError(f"{label}: incomplete simultaneous band.")
    joint_paths[label] = joint
    joint.to_csv(
        OUTPUT_DIR / f"{label.lower().replace('-', '_')}_joint_inference.csv",
        index=False,
    )

joint_paths["DRLPDID-IPT"].to_csv(
    OUTPUT_DIR / "drlpdid_ipt_inference_h19.csv", index=False
)
joint_paths["LPDID-RA"].to_csv(
    OUTPUT_DIR / "lpdid_ra_inference_h19.csv", index=False
)

dript_path = joint_paths["DRLPDID-IPT"]
fig, ax = plt.subplots(figsize=(10, 5.8))
ax.fill_between(
    dript_path["horizon"],
    dript_path["sim_ci_lower"],
    dript_path["sim_ci_upper"],
    color="#9ecae1",
    alpha=0.32,
    label="95% simultaneous sup-t band",
)
ax.fill_between(
    dript_path["horizon"],
    dript_path["ci_lower"],
    dript_path["ci_upper"],
    color="#3182bd",
    alpha=0.20,
    label="95% pointwise state-cluster-robust interval",
)
ax.plot(
    dript_path["horizon"],
    dript_path["estimate"],
    color="#08519c",
    marker="o",
    markersize=3.5,
    linewidth=1.7,
    label="DRLPDID-IPT",
)
ax.axhline(0, color="black", linewidth=0.9)
ax.axvline(-0.5, color="black", linestyle="--", linewidth=0.9)
ax.set(
    xlabel="Event-study horizon",
    ylabel="Estimated effect on female suicide mortality",
    title="No-fault divorce reforms: DRLPDID-IPT event study",
    xlim=(-H_PRE, H_POST),
)
ax.set_xticks(
    sorted(
        set(range(-H_PRE, H_POST + 1, 5))
        | {BASE_PERIOD, 0, H_POST}
    )
)
ax.legend(frameon=False, ncol=3)
fig.tight_layout()
fig.savefig(
    OUTPUT_DIR / "figure_drlpdid_ipt_h19.pdf",
    bbox_inches="tight",
)
fig.savefig(
    OUTPUT_DIR / "figure_drlpdid_ipt_h19.png",
    dpi=220,
    bbox_inches="tight",
)
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), sharex=True, sharey=True)
colors = {"LPDID-RA": "#d95f0e", "DRLPDID-IPT": "#08519c"}
for ax, mode in zip(axes, ["pointwise", "simultaneous"]):
    for label, path in joint_paths.items():
        lower = "ci_lower" if mode == "pointwise" else "sim_ci_lower"
        upper = "ci_upper" if mode == "pointwise" else "sim_ci_upper"
        ax.fill_between(path["horizon"], path[lower], path[upper],
                        color=colors[label], alpha=0.16)
        ax.plot(path["horizon"], path["estimate"], color=colors[label],
                linewidth=1.6, label=label)
    ax.axhline(0, color="black", linewidth=0.8)
    ax.axvline(-0.5, color="black", linestyle="--", linewidth=0.8)
    ax.set_title(
        "Pointwise 95% state-cluster-robust CIs"
        if mode == "pointwise"
        else "Simultaneous 95% multiplier bands"
    )
    ax.set_xlabel("Event-study horizon")
axes[0].set_ylabel("Effect on female suicide mortality")
axes[0].legend(frameon=False)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "figure_bacon_RA_DRIPT_inference.pdf", bbox_inches="tight")
fig.savefig(OUTPUT_DIR / "figure_bacon_RA_DRIPT_inference.png", dpi=220, bbox_inches="tight")
plt.show()

reference = local_paths["DRLPDID-IPT"]
local_comparators = [
    label for label, _, _ in LOCAL_SPECS
    if label != "DRLPDID-IPT"
]
fig, axes = plt.subplots(4, 2, figsize=(13, 16), sharex=True)
for ax, comparator in zip(axes.ravel(), local_comparators):
    other = local_paths[comparator]
    if other[
        ["estimate", "ci_lower", "ci_upper"]
    ].isna().any().any():
        raise RuntimeError(
            f"{comparator}: incomplete pointwise inference."
        )
    ax.fill_between(
        other["horizon"], other["ci_lower"], other["ci_upper"],
        color="#fdae6b", alpha=0.24, linewidth=0, zorder=1,
    )
    ax.fill_between(
        reference["horizon"],
        reference["ci_lower"],
        reference["ci_upper"],
        color="#6baed6", alpha=0.24, linewidth=0, zorder=2,
    )
    ax.plot(
        reference["horizon"], reference["estimate"],
        color="#08519c", linewidth=1.6,
        label="DRLPDID-IPT", zorder=4,
    )
    ax.plot(
        other["horizon"], other["estimate"],
        color="#d95f0e", linewidth=1.4, marker="o",
        markersize=2.5, label=comparator, zorder=4,
    )
    ax.axhline(0, color="black", linewidth=0.8)
    ax.axvline(-0.5, color="black", linestyle="--", linewidth=0.8)
    ax.set_title(f"DRLPDID-IPT vs. {comparator}")
    ax.legend(frameon=False, fontsize=8)
for ax in axes.ravel()[len(local_comparators):]:
    ax.set_visible(False)
for ax in axes[-1, :]:
    ax.set_xlabel("Event-study horizon")
for ax in axes[:, 0]:
    ax.set_ylabel("Estimated effect on female suicide mortality")
fig.suptitle(
    "No-fault divorce reforms: local-estimator comparisons\n"
    "Shaded areas show pointwise 95% state-cluster-robust intervals.",
    y=1.01,
)
fig.tight_layout(rect=(0, 0, 1, 0.975))
fig.savefig(
    OUTPUT_DIR / "figure_local_dynamic_comparisons_h19.pdf",
    bbox_inches="tight",
)
fig.savefig(
    OUTPUT_DIR / "figure_local_dynamic_comparisons_h19.png",
    dpi=220,
    bbox_inches="tight",
)
plt.show()

## 6. Sensitivity to additional observed covariates

The optional expanded specification uses standardized versions of
per-capita income, homicide mortality, and divorce cases. The affine
rescaling preserves the linear span and is used only for numerical
conditioning. A failed fit is recorded and is never replaced by a
fallback estimator.

In [ ]:
richer_result = None
richer_error = None
richer_inference_certified = False

if RUN_RICHER_SPECIFICATION:
    try:
        richer_result = fit_local(
            "DRLPDID-IPT richer",
            "dr-ipt",
            True,
            inference="cluster",
            covariates_override=RICHER_COVARIATES,
        )
        richer_event = validate_path(
            "DRLPDID-IPT richer", richer_result
        )
        richer_scalar = extract_scalar(
            "DRLPDID-IPT richer", richer_result, richer_event
        )
        baseline_scalar = local_table.loc[
            local_table["estimator"].eq("DRLPDID-IPT")
        ].iloc[0].to_dict()
        richer_comparison = pd.DataFrame([
            {
                "specification": "baseline_asmrh",
                **baseline_scalar,
            },
            {
                "specification": (
                    "richer_pcinc_asmrh_cases_standardized"
                ),
                **richer_scalar,
            },
        ])
        richer_comparison.to_csv(
            OUTPUT_DIR / "robustness_richer_h19.csv",
            index=False,
        )
        richer_event.assign(
            specification="richer_standardized"
        ).to_csv(
            OUTPUT_DIR / "event_study_richer_h19.csv",
            index=False,
        )

        fig, ax = plt.subplots(figsize=(10, 5.5))
        ax.plot(
            local_paths["DRLPDID-IPT"]["horizon"],
            local_paths["DRLPDID-IPT"]["estimate"],
            label="Baseline: homicide mortality",
            color="#08519c",
            linewidth=1.8,
        )
        ax.plot(
            richer_event["horizon"],
            richer_event["estimate"],
            label=(
                "Expanded: income + homicide mortality "
                "+ divorce cases"
            ),
            color="#d95f0e",
            linewidth=1.6,
        )
        ax.axhline(0, color="black", linewidth=0.8)
        ax.axvline(
            -0.5, color="black", linestyle="--", linewidth=0.8
        )
        ax.set(
            xlabel="Event-study horizon",
            ylabel="Estimated effect on female suicide mortality",
            title=(
                "DRLPDID-IPT: sensitivity to the "
                "covariate specification"
            ),
            xlim=(-H_PRE, H_POST),
        )
        ax.legend(frameon=False)
        fig.tight_layout()
        fig.savefig(
            OUTPUT_DIR / "figure_robustness_richer_h19.pdf",
            bbox_inches="tight",
        )
        fig.savefig(
            OUTPUT_DIR / "figure_robustness_richer_h19.png",
            dpi=220,
            bbox_inches="tight",
        )
        plt.show()
        display(richer_comparison)
        richer_inference_certified = True
    except Exception as exc:
        richer_error = f"{type(exc).__name__}: {exc}"
        warnings.warn(
            "The expanded specification was not certified. "
            f"No fallback was used. Detail: {richer_error}",
            RuntimeWarning,
        )
richer_status = {
    "requested": RUN_RICHER_SPECIFICATION,
    "raw_covariates": RICHER_COVARIATES_RAW,
    "model_covariates": RICHER_COVARIATES,
    "affine_standardization": True,
    "inference_certified": richer_inference_certified,
    "error": richer_error,
}
(OUTPUT_DIR / "richer_specification_status.json").write_text(
    json.dumps(richer_status, indent=2), encoding="utf-8"
)

## 7. Literature benchmarks (`diff_diff`)

This optional block reports only inference declared by the fitted
benchmark objects. Missing native standard errors or intervals remain
missing; the notebook does not construct substitutes.

In [ ]:
def external_event_frame(result, label):
    values = getattr(result, "event_study_effects", None)
    if isinstance(values, dict):
        rows = []
        for horizon, item in values.items():
            ci = item.get("conf_int", (np.nan, np.nan))
            cband = item.get("cband_conf_int", (np.nan, np.nan))
            rows.append({
                "horizon": int(horizon),
                "estimate": float(item["effect"]),
                "se": float(item.get("se", np.nan)),
                "ci_lower": float(ci[0]), "ci_upper": float(ci[1]),
                "sim_ci_lower": float(cband[0]), "sim_ci_upper": float(cband[1]),
            })
        frame = pd.DataFrame(rows)
    elif isinstance(getattr(result, "event_study", None), pd.DataFrame):
        frame = result.event_study.copy()
        rename = {}
        for candidate in ["event_time", "relative_time", "h"]:
            if candidate in frame:
                rename[candidate] = "horizon"; break
        for candidate in ["effect", "att", "coef"]:
            if candidate in frame:
                rename[candidate] = "estimate"; break
        for candidate in ["std_error", "standard_error", "stderr"]:
            if candidate in frame:
                rename[candidate] = "se"; break
        frame.rename(columns=rename, inplace=True)
    else:
        raise TypeError(f"{label}: unrecognized event-study output.")
    if not {"horizon", "estimate"}.issubset(frame):
        raise ValueError(f"{label}: horizon/estimate columns are absent.")
    for column in ["se", "ci_lower", "ci_upper", "sim_ci_lower", "sim_ci_upper"]:
        if column not in frame:
            frame[column] = np.nan
        frame[column] = pd.to_numeric(frame[column], errors="coerce")
    frame["horizon"] = pd.to_numeric(frame["horizon"], errors="raise").astype(int)
    frame["estimate"] = pd.to_numeric(frame["estimate"], errors="raise")
    if -1 not in set(frame["horizon"]):
        frame = pd.concat([frame, pd.DataFrame([{
            "horizon": -1, "estimate": 0.0, "se": 0.0,
            "ci_lower": 0.0, "ci_upper": 0.0,
            "sim_ci_lower": 0.0, "sim_ci_upper": 0.0,
        }])], ignore_index=True)
    frame.loc[frame["horizon"].eq(-1),
              ["estimate", "se", "ci_lower", "ci_upper",
               "sim_ci_lower", "sim_ci_upper"]] = 0.0
    return frame.sort_values("horizon").reset_index(drop=True)

def native_overall(result, label):
    ci = getattr(result, "overall_conf_int")
    row = {
        "estimator": label,
        "estimate": float(getattr(result, "overall_att")),
        "se": float(getattr(result, "overall_se")),
        "ci_lower": float(ci[0]),
        "ci_upper": float(ci[1]),
        "estimand": "diff_diff native overall ATT",
    }
    if not np.isfinite(list(row.values())[1:5]).all():
        raise ValueError(f"{label}: incomplete native overall inference.")
    return row

external_paths = {}
external_results = {}
external_scalars = []
external_errors = {}
if RUN_EXTERNAL_BENCHMARKS and HAVE_DIFFDIFF:
    external_data = df[["id", "year", "y", "g", *BASELINE_COVARIATES]].copy()
    cs_kwargs = dict(
        control_group="not_yet_treated", base_period="universal",
        anticipation=0, cluster="id", n_bootstrap=0, seed=SEED,
    )
    try:
        cs_supports_cband = (
            "cband" in inspect.signature(
                CallawaySantAnna
            ).parameters
        )
        if cs_supports_cband:
            cs_kwargs["cband"] = False
    except (TypeError, ValueError):
        cs_supports_cband = False
    constructors = {
        "Callaway--Sant'Anna": lambda: CallawaySantAnna(**cs_kwargs).fit(
            external_data, outcome="y", unit="id", time="year",
            first_treat="g", covariates=BASELINE_COVARIATES,
            aggregate="event_study"),
        "Borusyak--Jaravel--Spiess": lambda: ImputationDiD(
            anticipation=0, cluster="id", n_bootstrap=0, seed=SEED,
            aux_partition="cohort_horizon", pretrends=True).fit(
            external_data, outcome="y", unit="id", time="year",
            first_treat="g", covariates=BASELINE_COVARIATES,
            aggregate="event_study"),
        "Sun--Abraham": lambda: SunAbraham(
            control_group="not_yet_treated", anticipation=0,
            cluster="id", n_bootstrap=0, seed=SEED).fit(
            external_data, outcome="y", unit="id", time="year",
            first_treat="g", covariates=BASELINE_COVARIATES),
        "Gardner": lambda: TwoStageDiD(
            anticipation=0, cluster="id", n_bootstrap=0, seed=SEED,
            pretrends=True).fit(
            external_data, outcome="y", unit="id", time="year",
            first_treat="g", covariates=BASELINE_COVARIATES,
            aggregate="event_study"),
    }
    for label, constructor in constructors.items():
        try:
            fitted = constructor()
            external_results[label] = fitted
            external_paths[label] = external_event_frame(fitted, label)
            external_scalars.append(native_overall(fitted, label))
        except Exception as exc:
            external_errors[label] = f"{type(exc).__name__}: {exc}"

    if (
        "Callaway--Sant'Anna" in external_paths
        and cs_supports_cband
    ):
        print("Estimating Callaway--Sant'Anna simultaneous band")
        try:
            cs_band_result = CallawaySantAnna(
                control_group="not_yet_treated",
                base_period="universal",
                anticipation=0,
                cluster="id",
                n_bootstrap=N_MULTIPLIER,
                seed=SEED,
                cband=True,
            ).fit(
                external_data,
                outcome="y",
                unit="id",
                time="year",
                first_treat="g",
                covariates=BASELINE_COVARIATES,
                aggregate="event_study",
            )
            cs_band_path = external_event_frame(
                cs_band_result,
                "Callaway--Sant'Anna simultaneous band",
            )
            cs_pointwise_path = external_paths[
                "Callaway--Sant'Anna"
            ]
            comparison = cs_pointwise_path[
                ["horizon", "estimate"]
            ].merge(
                cs_band_path[["horizon", "estimate"]],
                on="horizon",
                suffixes=("_cluster", "_multiplier"),
                validate="one_to_one",
            )
            maximum_cs_difference = float(
                np.max(
                    np.abs(
                        comparison["estimate_cluster"]
                        - comparison["estimate_multiplier"]
                    )
                )
            )
            if maximum_cs_difference > 1e-8:
                raise RuntimeError(
                    "Callaway--Sant'Anna point estimates changed "
                    "across inference modes: "
                    f"{maximum_cs_difference:.3e}."
                )
            external_paths["Callaway--Sant'Anna"] = (
                cs_pointwise_path.drop(
                    columns=["sim_ci_lower", "sim_ci_upper"]
                ).merge(
                    cs_band_path[
                        [
                            "horizon",
                            "sim_ci_lower",
                            "sim_ci_upper",
                        ]
                    ],
                    on="horizon",
                    how="left",
                    validate="one_to_one",
                )
            )
        except Exception as exc:
            external_errors[
                "Callaway--Sant'Anna simultaneous band"
            ] = f"{type(exc).__name__}: {exc}"
            warnings.warn(
                "The Callaway--Sant'Anna pointwise inference "
                "remains available, but its native simultaneous "
                "band could not be certified.",
                RuntimeWarning,
            )

    if external_paths:
        external_event_all = pd.concat(
            [path.assign(estimator=label) for label, path in external_paths.items()],
            ignore_index=True,
        )
        external_event_all.to_csv(
            OUTPUT_DIR / "external_event_studies.csv", index=False
        )
        external_event_all.to_csv(
            OUTPUT_DIR / "external_event_studies_h19.csv",
            index=False,
        )
    if external_scalars:
        external_table = pd.DataFrame(external_scalars)
        external_table.to_csv(
            OUTPUT_DIR / "table_external_native_att.csv", index=False
        )
        table_all_estimators = pd.concat(
            [
                local_table.assign(
                    estimand="equal-horizon average h=0,...,19",
                    source="pydrlpdid native joint influence function",
                ),
                external_table.assign(
                    window="native",
                    source="diff_diff native overall inference",
                ),
            ],
            ignore_index=True,
            sort=False,
        )
        table_all_estimators.to_csv(
            OUTPUT_DIR / "table_all_estimators_h19.csv",
            index=False,
        )

    needed_for_figure = {
        "Callaway--Sant'Anna",
        "Borusyak--Jaravel--Spiess",
        "Sun--Abraham",
        "Gardner",
    }
    if needed_for_figure.issubset(external_paths):
        local_rw_path = local_paths["LPDID-RW+X"][
            ["horizon", "estimate", "ci_lower", "ci_upper"]
        ].copy()
        local_rw_path["sim_ci_lower"] = np.nan
        local_rw_path["sim_ci_upper"] = np.nan
        six_comparators = [
            ("LPDID-RW+X", local_rw_path),
            ("LPDID-RA", joint_paths["LPDID-RA"]),
            *[
                (label, external_paths[label])
                for label in [
                    "Callaway--Sant'Anna",
                    "Borusyak--Jaravel--Spiess",
                    "Sun--Abraham",
                    "Gardner",
                ]
            ],
        ]
        reference_joint = joint_paths["DRLPDID-IPT"]
        availability_rows = []
        for label, path in [
            ("DRLPDID-IPT", reference_joint),
            *six_comparators,
        ]:
            window = path.loc[
                path["horizon"].between(-H_PRE, H_POST)
            ]
            has_band = not window[
                ["sim_ci_lower", "sim_ci_upper"]
            ].isna().any().any()
            availability_rows.append({
                "estimator": label,
                "pointwise_interval": (
                    "native state-cluster-robust interval"
                ),
                "simultaneous_band": (
                    "native state-clustered simultaneous band"
                    if has_band else "not exposed by fitted result"
                ),
            })
        inference_availability = pd.DataFrame(availability_rows)
        inference_availability.to_csv(
            OUTPUT_DIR
            / "event_study_inference_availability_h19.csv",
            index=False,
        )

        fig, axes = plt.subplots(
            3, 2, figsize=(13, 13), sharex=True
        )
        required_horizons = set(range(-H_PRE, H_POST + 1))
        for ax, (label, other) in zip(
            axes.ravel(), six_comparators
        ):
            other_window = other.loc[
                other["horizon"].between(-H_PRE, H_POST),
                [
                    "horizon",
                    "estimate",
                    "ci_lower",
                    "ci_upper",
                    "sim_ci_lower",
                    "sim_ci_upper",
                ],
            ].copy()
            present = set(other_window["horizon"].astype(int))
            if present != required_horizons:
                raise RuntimeError(
                    f"{label}: incomplete plotted window; "
                    f"missing {sorted(required_horizons-present)}."
                )
            if other_window[
                ["estimate", "ci_lower", "ci_upper"]
            ].isna().any().any():
                raise RuntimeError(
                    f"{label}: incomplete native pointwise inference."
                )
            comparator_has_band = not other_window[
                ["sim_ci_lower", "sim_ci_upper"]
            ].isna().any().any()
            if comparator_has_band:
                ax.fill_between(
                    other_window["horizon"],
                    other_window["sim_ci_lower"],
                    other_window["sim_ci_upper"],
                    color="#fdd0a2",
                    alpha=0.34,
                    linewidth=0,
                    zorder=1,
                )
            ax.fill_between(
                reference_joint["horizon"],
                reference_joint["sim_ci_lower"],
                reference_joint["sim_ci_upper"],
                color="#c6dbef",
                alpha=0.34,
                linewidth=0,
                zorder=1,
            )
            ax.fill_between(
                other_window["horizon"],
                other_window["ci_lower"],
                other_window["ci_upper"],
                color="#f16913",
                alpha=0.20,
                linewidth=0,
                zorder=2,
            )
            ax.fill_between(
                reference_joint["horizon"],
                reference_joint["ci_lower"],
                reference_joint["ci_upper"],
                color="#4292c6",
                alpha=0.20,
                linewidth=0,
                zorder=2,
            )
            ax.plot(
                reference_joint["horizon"],
                reference_joint["estimate"],
                color="#08519c",
                linewidth=1.7,
                label="DRLPDID-IPT",
                zorder=4,
            )
            ax.plot(
                other_window["horizon"],
                other_window["estimate"],
                color="#d95f0e",
                linewidth=1.4,
                label=label,
                zorder=4,
            )
            ax.axhline(0, color="black", linewidth=0.8)
            ax.axvline(
                -0.5,
                color="black",
                linestyle="--",
                linewidth=0.8,
            )
            ax.set_xlim(-H_PRE, H_POST)
            ax.set_title(f"DRLPDID-IPT vs. {label}")
            ax.legend(frameon=False, fontsize=8)
            availability_note = (
                "Both estimators: pointwise interval + "
                "simultaneous band"
                if comparator_has_band
                else f"{label}: pointwise interval only"
            )
            ax.text(
                0.02, 0.02, availability_note,
                transform=ax.transAxes,
                fontsize=7.5,
                color="#4d4d4d",
                va="bottom",
            )
        for ax in axes[-1, :]:
            ax.set_xlabel("Event-study horizon")
        for ax in axes[:, 0]:
            ax.set_ylabel(
                "Estimated effect on female suicide mortality"
            )
        fig.suptitle(
            "No-fault divorce reforms: estimator comparisons\n"
            "Dark shading: pointwise 95% state-cluster-robust "
            "intervals; light shading: simultaneous 95% "
            "state-clustered bands where available.",
            y=1.01,
        )
        fig.tight_layout(rect=(0, 0, 1, 0.975))
        fig.savefig(
            OUTPUT_DIR / "figure_dynamic_comparisons_h19.pdf",
            bbox_inches="tight",
        )
        fig.savefig(
            OUTPUT_DIR / "figure_dynamic_comparisons_h19.png",
            dpi=220,
            bbox_inches="tight",
        )
        plt.show()
elif RUN_EXTERNAL_BENCHMARKS:
    external_errors["import"] = DIFFDIFF_ERROR

(OUTPUT_DIR / "external_benchmark_errors.json").write_text(
    json.dumps(external_errors, indent=2), encoding="utf-8"
)
print("External benchmark errors:", external_errors)
display(pd.DataFrame(external_scalars))

## 8. Reproducibility manifest and final checks

In [ ]:
dependency_versions = {}
for package in [
    "numpy",
    "pandas",
    "scipy",
    "statsmodels",
    "patsy",
    "matplotlib",
]:
    try:
        dependency_versions[package] = (
            importlib_metadata.version(package)
        )
    except importlib_metadata.PackageNotFoundError:
        dependency_versions[package] = None

NOTEBOOK_PATH = (
    PROJECT_DIR
    / "notebooks"
    / "Application_1_Bacon_pydrlpdid_v0.7.2_LOCAL_H19.ipynb"
)
notebook_source_sha256 = None
if NOTEBOOK_PATH.exists():
    notebook_document = json.loads(
        NOTEBOOK_PATH.read_text(encoding="utf-8")
    )
    source_payload = "\n".join(
        cell.get("cell_type", "")
        + "\n"
        + "".join(cell.get("source", []))
        for cell in notebook_document.get("cells", [])
    )
    notebook_source_sha256 = hashlib.sha256(
        source_payload.encode("utf-8")
    ).hexdigest()
output_files = sorted(
    set(
        path.name
        for path in OUTPUT_DIR.iterdir()
        if path.is_file()
    )
    | {"manifest.json", "run_manifest.json"}
)
manifest = {
    "application": "Bacon no-fault-divorce reforms",
    "package_version": pydrlpdid.__version__,
    "package_file": str(PACKAGE_FILE),
    "package_core_file": str(PACKAGE_CORE_FILE),
    "package_core_sha256": sha256_file(PACKAGE_CORE_FILE),
    "notebook_file": str(NOTEBOOK_PATH),
    "notebook_source_sha256": notebook_source_sha256,
    "python": sys.version,
    "platform": platform.platform(),
    "dependency_versions": dependency_versions,
    "data_sha256": sha256_file(DATA_PATH),
    "horizons": [-H_PRE, H_POST],
    "post_window": list(POST_WINDOW),
    "covariates": BASELINE_COVARIATES,
    "richer_specification": richer_status,
    "formal_estimators_common_supported_stack": True,
    "lpdid_ra_native_dube_sample": True,
    "propensity_clipping": False,
    "fallback_estimator": False,
    "cluster": "state",
    "multiplier_draws": N_MULTIPLIER,
    "seed": SEED,
    "diff_diff_version": DIFFDIFF_VERSION,
    "external_benchmark_errors": external_errors,
    "outputs": output_files,
}
for manifest_name in ["manifest.json", "run_manifest.json"]:
    (OUTPUT_DIR / manifest_name).write_text(
        json.dumps(manifest, indent=2), encoding="utf-8"
    )
print(json.dumps(manifest, indent=2))

## Guide to the main output files

- `table_local_h19.csv/.tex`: local scalar results over
  \(h=0,\ldots,19\), with native joint influence-function inference.
- `table_external_native_att.csv`: package-native overall ATT,
  standard error, and confidence interval returned by `diff_diff`.
- `local_event_study_h19.csv` and `external_event_studies_h19.csv`:
  complete dynamic paths and native pointwise inference.
- `figure_drlpdid_ipt_h19.pdf/.png`: standalone DRLPDID-IPT path with
  pointwise and simultaneous inference.
- `figure_bacon_RA_DRIPT_inference.pdf/.png`: focused comparison of
  LPDID-RA and DRLPDID-IPT.
- `figure_local_dynamic_comparisons_h19.pdf/.png`: comparisons among
  the eight local estimators.
- `figure_dynamic_comparisons_h19.pdf/.png`: DRLPDID-IPT against the
  selected local and staggered-adoption benchmarks.
- `nested_basis_identity_audit.csv`, `nuisance_rank_audit.csv`, and
  `local_validation.json`: formal numerical certification.
- `manifest.json`: frozen versions, hashes, specification, and output
  inventory.